# Veritas AI Heart-Disease Risk Stratification MVP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erjanowitz-code/veritas-heart-risk-mvp/blob/main/veritas_heart_risk.ipynb)

> **Prototype, not medical advice.** This is a student hackathon project trained on 303 patients from one
> clinic in 1988. It has not been clinically validated. Do not use it to make health decisions; talk to a doctor.

**Our project takes a patient's clinical measurements and gives back a Low / Moderate / High heart-disease risk tier.**

How to use this notebook: **Runtime → Run all**. The last cell starts an interactive demo that asks for patient
vitals in a text box. The code lives in the GitHub repo
[erjanowitz-code/veritas-heart-risk-mvp](https://github.com/erjanowitz-code/veritas-heart-risk-mvp).
The first cell downloads it.

## 1. Setup
Downloads the project code from GitHub and installs the packages it needs.

In [ ]:
import os

REPO = "https://github.com/erjanowitz-code/veritas-heart-risk-mvp.git"
REPO_DIR = "veritas-heart-risk-mvp"

if not os.path.exists("model.py"):  # not already inside the project folder
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO}
    os.chdir(REPO_DIR)
print("Working in:", os.getcwd())

%pip install -q -r requirements.txt

## 2. Data
The Cleveland subset of the UCI Heart Disease dataset, loaded from Hugging Face (`skrishna/heart_disease_uci`).
`target` = 1 means heart disease is present. Missing values (`?`) become NaN and are filled later inside
the model Pipeline using **training-set** medians only, so the test set never leaks into training.

In [ ]:
from data_loader import FEATURE_INFO, clean, load_raw

df = clean(load_raw())
print(f"{len(df)} patients, {df.shape[1] - 1} features")
print("Missing values:", df.isna().sum()[df.isna().sum() > 0].to_dict())
print("Patients with heart disease:", int(df['target'].sum()), f"({df['target'].mean():.0%})")
df.head()

## 3. Baseline vs Random Forest, and fairness check
Trains a `DummyClassifier` baseline (always predicts the most common class) and a Random Forest, then reports
accuracy, precision, **recall** (how many sick patients we catch) and confusion matrices on 61 held-out patients.
It also runs 5-fold cross-validation on the training set and breaks results down by sex and age group.

In [ ]:
import model

forest, X_train = model.main()

## 4. Charts
Saved to the `charts/` folder by the cell above.

In [ ]:
from IPython.display import Image, display

for name in ["confusion_matrix_baseline", "confusion_matrix_random_forest",
             "feature_importance", "fairness_by_group"]:
    print(name.replace("_", " ").title())
    display(Image(f"charts/{name}.png", width=520))

## 5. Interactive demo
Type a value for each vital and press Enter, or just press Enter to use the typical (median) value.
Type `q` to quit. Risk tiers: **Low** < 33% predicted probability, **Moderate** 33–66%, **High** > 66%
(prototype cut-offs, not clinical ones).

In [ ]:
from utils import interactive_loop

interactive_loop(forest, X_train)